This jupyter notebook enables an error propagation based on landmarks.
The inputs encompass the landmarks and transformation matrices.
The landmarks were selected in napari using the affinder plug-in which also determined the transformation matrices (column Matrices in the input table).
In the rows where the ROI is called 'vis' the Matrix refers to the affine transformation matrix between a 10X H&E of the visium and the AFM section.
In all other rows the Matrix refers to the affine transformation matrix between the 20X Hoechst image and the 20X H&E image of the AFM section.
The template matching matrices were determined using the "Mapping_withConfigFile.ipynb" script.

For Patient 1, there was a rotation of 18 degrees between the 20X H&E image and the 10X H&E tilescan of the AFM section. To accomodate this, a rotation matrix was multiplied with
the template matchingmatrices determined by the "Mapping_withConfigFile.ipynb" script.

rotation_matrix = np.array([[ 9.51056516e-01,  3.09016994e-01, -1.26665615e+03],

                            [-3.09016994e-01,  9.51056516e-01,  1.40164784e+03],
       
                            [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

This correction is already reflected in the input table.

In [5]:
# import libraries and define functions
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import openpyxl
from pathlib import Path

# extract of landmarks from excel file
def transform_point_entry_to_array_list(df,roi,index_name):
    landmarks = df[df['ROI']==roi][index_name].tolist()
    landmarks = [np.fromstring(pts.strip('[]\n'),sep=' ') for pts in landmarks]
    return landmarks

# extract matrix from excel file
def transform_matrix_entry_to_matrix(df,roi,column):
    matrix = df[df['ROI']==roi][column].tolist()[0]
    matrix = np.fromstring(matrix.replace('[','').replace(']',''),sep=' ').reshape(3,3)

    #index array to re-order the matrix so that it can be applied to points instead of images
    index_array = np.array([3,2,1,0,5,4])
    return np.hstack([matrix[:2,:2].reshape(-1),matrix[:2,2]])[index_array]

def transform_matrix_entry_to_affine_transform_matrix(df,roi,column):
    matrix = df[df['ROI']==roi][column].tolist()[0]
    matrix = np.fromstring(matrix.replace('[','').replace(']',''),sep=' ').reshape(3,3)
    return matrix

# calculate the covariance ('error') of the matrix parameter
def calculate_parameter_covariance(pts_src,residuals):
    pts_src = np.asarray(pts_src)
    residuals = np.asarray(residuals)
    n = pts_src.shape[0]

    #create Jacobi matrix from partial derivatives
    #x' = ax + by + e
    #y' = cy + dy + f
    # even rows are derivatives of x, odd rows are derivatives of y
    # the columns differ in the variable (a-f) from which they were derived
    J = np.zeros((2*n, 6))
    for i, (xs, ys) in enumerate(pts_src):
        J[2*i]   = [xs, ys, 0, 0, 1, 0]
        J[2*i+1] = [0, 0, xs, ys, 0, 1]
    dof = max(1, 2*n - 6)  # degrees of freedom
    sigma2_hat = (residuals @ residuals) / dof
    # Covariance of parameters: sigma2 * (J^T J)^{-1}
    # Use pseudo-inverse in case J^T J near-singular
    JTJ = J.T.dot(J)
    inv_JTJ = np.linalg.pinv(JTJ)
    Sigma_p = sigma2_hat * inv_JTJ
    return Sigma_p

# transform a point into specific format for calculation
def Jp_affine_at_point(x):
    x1, x2 = x
    Jp = np.array([[x1, x2, 0, 0, 1, 0],
                   [0, 0, x1, x2, 0, 1]])
    return Jp

# Propagate a point trough all transforms and also propagate the error along with it
def propagate_point_A_to_D(xA, affine1, Sigma_p1, translation, affine2, Sigma_p2):
    ## define matrices
    # first affine transformation
    A1 = affine1[:4].reshape(2,2)
    t1 = affine1[4:]
    # translation
    Ts = translation[:4].reshape(2,2)
    ts = translation[4:]
    # second affine transformation
    A2 = affine2[:4].reshape(2,2)
    t2 = affine2[4:]

    ## propagate the point
    # first transform
    xB = A1.dot(xA) + t1
    # second transform
    xC = Ts.dot(xB) + ts
    # third transform
    xD = A2.dot(xC) + t2

    ## propagate the error
    # first transform
    Jp1 = Jp_affine_at_point(xA)
    Sigma_B = Jp1.dot(Sigma_p1).dot(Jp1.T)
    # second transform (no propagation only scaling)
    Sigma_C = Ts.dot(Sigma_B).dot(Ts.T)
    # third transform
    # uncertainty just for last last transform
    Jp2 = Jp_affine_at_point(xC)
    Sigma_from_CD_params = Jp2.dot(Sigma_p2).dot(Jp2.T)
    # actual propagation
    Sigma_from_prev = A2.dot(Sigma_C).dot(A2.T)
    # combined error
    Sigma_D = Sigma_from_CD_params + Sigma_from_prev
    return xD, Sigma_D

def affine6_to_matrix3x3(arr):
    """Convert 6-element affine array to 3x3 matrix."""
    a, b, c, d, tx, ty = arr
    return np.array([
        [a, b, tx],
        [c, d, ty],
        [0, 0, 1]
    ])

In [14]:
# import excel files with landmarks and transformation matrices and points to transform

path_landmarks = Path('../Inputs/Landmarks.xlsx')
data_landmarks = pd.read_excel(path_landmarks)
data_landmarks = data_landmarks.iloc[:-1]
path_afm_points = Path('../Inputs/AFM_Coords.xlsx')
data_afm_points = pd.read_excel(path_afm_points)

# import the AFM measurement table to merge the propagated error to stiffness values
path_measurements = Path('../Inputs/AFM_Measurements.xlsx')
measurements_df = pd.read_excel(path_measurements)

In [15]:
# get the different sections
sections = data_landmarks['section'].unique()

In [11]:
# iterate over each section and roi to get matrix parameter covariance
cov_p = dict()
matrices = dict()
tm_matrices = dict()
for section in sections:
    sub_cov_p = dict()
    sub_matrices = dict()
    sub_translation = dict()
    df = data_landmarks[data_landmarks['section']==section]
    rois = df['ROI'].unique()
    for roi in rois:
        matrix = transform_matrix_entry_to_matrix(df,roi,'Matrix')
        pts_src = transform_point_entry_to_array_list(df,roi,'to_be_transformed')
        residuals = transform_point_entry_to_array_list(df,roi,'difference_single')
        residuals = np.hstack(residuals)
        sub_cov_p[roi] = calculate_parameter_covariance(pts_src,residuals)
        sub_matrices[roi] = matrix
        if roi != 'vis':
            matrix = transform_matrix_entry_to_matrix(df,roi,'tm_matrix')
            sub_translation[roi] = matrix
    cov_p[section] = sub_cov_p
    matrices[section] = sub_matrices
    tm_matrices[section] = sub_translation

In [18]:
# perform transform the coordinates and propagate the error
df = []
for section in sections:
    afm_df = data_afm_points[data_afm_points['section']==section]
    rois = afm_df['ROI'].unique().tolist()
    for roi in rois:
        coords = transform_point_entry_to_array_list(afm_df,roi,'AFM_ROI_coords')
        #keep the position index for latter mapping to the stiffness values
        position_indices = afm_df[afm_df['ROI'] == roi]['Position_Index'].to_list() 
        for xA, pi in zip(coords, position_indices):
            affine1 = matrices[section][roi]
            Sigma_p1 = cov_p[section][roi]
            translation = tm_matrices[section][roi]
            affine2 = matrices[section]['vis']
            Sigma_p2 = cov_p[section]['vis']
            xt,sigma_D = propagate_point_A_to_D(xA, affine1, Sigma_p1, translation, affine2, Sigma_p2)
            std_x, std_y = np.sqrt(np.diag(sigma_D))
            df.append(pd.DataFrame({'section':[section],'ROI':[roi],
                                    'Position_Index': pi,
                                    'x_start':xA[0],'y_start':xA[1],
                                    'x_final':[xt[0]],'y_final':[xt[1]],'std_x':[std_x],'std_y':[std_y]}))
df = pd.concat(df)
df

,section,ROI,Position_Index,x_start,y_start,x_final,y_final,std_x,std_y
0,Patient01,ROI01,0,1293.703326,1293.703326,8318.764646,5674.291095,20.308610,20.315291
0,Patient01,ROI01,1,1293.703326,1233.769254,8332.606856,5652.763587,20.427974,20.433518
0,Patient01,ROI01,2,1293.703326,1173.835181,8346.449066,5631.236078,20.549676,20.554229
0,Patient01,ROI01,3,1293.703326,1113.901109,8360.291276,5609.708570,20.673675,20.677383
0,Patient01,ROI01,4,1293.703326,1053.967036,8374.133486,5588.181061,20.799931,20.802935
...,...,...,...,...,...,...,...,...,...
0,Patient7a,ROI02,94,754.296674,994.032964,9087.951666,7723.219157,17.290192,17.286205
0,Patient7a,ROI02,95,754.296674,1053.967036,9062.410990,7718.407155,17.085280,17.081220
0,Patient7a,ROI02,96,754.296674,1113.901109,9036.870315,7713.595152,16.890121,16.885807
0,Patient7a,ROI02,98,754.296674,1233.769254,8985.788963,7703.971146,16.530424,16.525038


In [17]:
measurements_df

,Position_Index,X_Position,Y_Position,Baseline_Offset_[N],Baseline_Slope_[N/m],Contact_Point_Offset_[m],Young's_modulus_[Pa],Contact_Point_[m],Baseline_[N],Discontinuity_Position,Discontinuity_Width_[m],Adhesion_[N],ResidualRMS_[N],ROI,Sample
0,0,0.001546,0.000690,-3.500000e-09,0.000021,0.000067,4994.1300,-6.500000e-08,5.600000e-12,NaN,0,0,3.030000e-11,ROI01,Patient7b
1,1,0.001566,0.000690,-4.400000e-09,0.000028,0.000067,2826.0000,-1.500000e-08,2.250000e-12,NaN,0,0,1.870000e-11,ROI01,Patient7b
2,2,0.001586,0.000690,-4.800000e-09,0.000028,0.000063,126.5360,8.070000e-08,1.490000e-12,NaN,0,0,8.590000e-12,ROI01,Patient7b
3,4,0.001626,0.000690,-1.200000e-09,-0.000043,0.000065,128.8860,6.870000e-08,-1.200000e-12,NaN,0,0,8.800000e-12,ROI01,Patient7b
4,5,0.001646,0.000690,-6.800000e-09,0.000036,0.000066,2482.6100,-3.000000e-08,1.370000e-11,NaN,0,0,2.070000e-11,ROI01,Patient7b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5242,20,0.000241,-0.001483,-4.131900e-09,-0.000018,0.000080,161.5030,6.941450e-08,-2.612530e-12,NaN,0,0,1.000730e-11,ROI06,Patient03
5243,21,0.000261,-0.001483,-2.013630e-09,-0.000044,0.000079,87.5699,1.562800e-07,-2.037700e-12,NaN,0,0,9.843600e-12,ROI06,Patient03
5244,22,0.000281,-0.001483,-3.551410e-09,-0.000026,0.000078,79.2489,1.381910e-07,-1.344320e-12,NaN,0,0,9.269290e-12,ROI06,Patient03
5245,23,0.000301,-0.001483,-4.026230e-09,-0.000020,0.000078,66.6360,1.223170e-07,-1.778930e-12,NaN,0,0,8.632060e-12,ROI06,Patient03
